In [17]:
import openai, json

client = openai.OpenAI()
messages = []

In [18]:
import requests

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"


def get_weather(city):
    return "33 degrees celcius."


def get_popular_movies():
    """인기 영화 목록을 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies")
    return json.dumps(resp.json(), ensure_ascii=False)


def get_movie(movie_id: int):
    """영화 ID로 영화 상세 정보를 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{movie_id}")
    return json.dumps(resp.json(), ensure_ascii=False)


def get_movie_credits(movie_id: int):
    """영화 ID로 출연진 및 제작진 정보를 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{movie_id}/credits")
    return json.dumps(resp.json(), ensure_ascii=False)


def get_movie_videos(movie_id: int):
    """영화 ID로 예고편/비디오 목록을 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{movie_id}/videos")
    return json.dumps(resp.json(), ensure_ascii=False)


def get_movie_providers(movie_id: int):
    """영화 ID로 스트리밍 제공처 정보를 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{movie_id}/providers")
    return json.dumps(resp.json(), ensure_ascii=False)


def get_similar_movies(movie_id: int):
    """영화 ID로 비슷한 영화 목록을 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{movie_id}/similar")
    return json.dumps(resp.json(), ensure_ascii=False)


FUNCTION_MAP = {
    "get_weather": get_weather,
    "get_popular_movies": get_popular_movies,
    "get_movie": get_movie,
    "get_movie_credits": get_movie_credits,
    "get_movie_videos": get_movie_videos,
    "get_movie_providers": get_movie_providers,
    "get_similar_movies": get_similar_movies,
}

In [19]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "A function to get the weather of a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the weather of.",
                    }
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기 영화 목록을 반환합니다.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie",
            "description": "영화 ID로 영화 상세 정보를 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "영화 ID",
                    }
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "영화 ID로 출연진 및 제작진 정보를 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "영화 ID",
                    }
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_videos",
            "description": "영화 ID로 예고편/비디오 목록을 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "영화 ID",
                    }
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_providers",
            "description": "영화 ID로 스트리밍 제공처 정보를 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "영화 ID",
                    }
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "영화 ID로 비슷한 영화 목록을 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "영화 ID",
                    }
                },
                "required": ["movie_id"],
            },
        },
    },
]

In [20]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [ ]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 인기있는 영화를 알려줘
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {} for a result of [{"adult": false, "backdrop_path": "https://image.tmdb.org/t/p/w1280/6YjnTRBz704LF1uJ3ZC4wsS9T8r.jpg", "genre_ids": [28, 80, 53], "id": 1290821, "original_language": "en", "original_title": "Shelter", "overview": "A man living in self-imposed exile on a remote island rescues a young girl from a violent storm, setting off a chain of events that forces him out of seclusion to protect her from enemies tied to his past.", "popularity": 456.5931, "poster_path": "https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg", "release_date": "2026-01-28", "title": "Shelter", "video": false, "vote_average": 6.925, "vote_count": 134}, {"adult": false, "backdrop_path": "https://image.tmdb.org/t/p/w1280/7HKpc11uQfxnw0Y8tRUYn1fsKqE.jpg", "genre_ids": [878, 28, 53], "id": 1236153, "original_language": "en", "original_title": "Mercy", "overview": "In the near future, a detective sta